<a href="https://colab.research.google.com/github/AnshuOnGit/ConsumeRESTHTTPClient/blob/master/vector_dbload_and_query.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install Dependencies

In [ ]:
!pip install langchain
!pip install transformers
!pip install PyPDF2
!pip install --upgrade langchain
!pip install langchain_community
!pip install sentence_transformers
!pip uninstall transformers
!pip install ipywidgets
!pip install pypdf
!pip install accelerate

Load Pdf to vector DB

In [ ]:
from langchain.document_loaders import PyPDFLoader
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from transformers import GPT2TokenizerFast
from langchain.text_splitter import RecursiveCharacterTextSplitter
import torch

#Inititalise the embedding
hf_embeddings = HuggingFaceEmbeddings()

#Load documents
loader = PyPDFLoader('750216638_1afcdc145e39400a9e317f4c64c179f7-100624-0453-216.pdf')
pages = loader.load()

from transformers import BertTokenizer, BertModel

#model_name = 'VMware/vbert-2021-base'
#tokenizer = BertTokenizer.from_pretrained(model_name)
#model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype= torch.float16, device_map = 'sequential', offload_folder='/Users/kanshu/chat-with-my-docs/chat-app/offload_folder')

#Split the token
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
text_split = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(tokenizer, chunk_size=1800, chunk_overlap=50)
text = text_split.split_documents(pages)

#Create the vectorstore
store = Chroma.from_documents(text,hf_embeddings,persist_directory='saved_vdb5')
store.persist()

/Users/kanshu/chat-with-my-docs/chat-app/lib/python3.11/site-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 0.3.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  warn_deprecated(
/Users/kanshu/chat-with-my-docs/chat-app/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid usin

Query Vector DB

In [ ]:

def query(prompt):
    #Load the vectorstore
    vectordb = Chroma(persist_directory='saved_vdb5', embedding_function=hf_embeddings)


    #Get the semantic paragraph
    #prompt = 'how to onboard storage backend'
    search_results = vectordb.similarity_search_with_score(prompt)
    print("=========vector db output =====>")
    print(search_results)
    context = ' '.join([result[0].page_content for result in search_results])
    #print("=========context is =====>")
    #print(context)
    return context




In [ ]:
prompt = "how to create datastore"
query(prompt)

=========vector db output =====>
[(Document(page_content='Volumes which will be created as part of datastore creation will be shown in table.', metadata={'page': 88, 'source': '750216638_1afcdc145e39400a9e317f4c64c179f7-100624-0453-216.pdf'}), 0.6482599377632141), (Document(page_content='Volumes which will be created as part of datastore creation will be shown in table.', metadata={'page': 88, 'source': '750216638_1afcdc145e39400a9e317f4c64c179f7-100624-0453-216.pdf'}), 0.6482599377632141), (Document(page_content='Volumes which will be created as part of datastore creation will be shown in table.', metadata={'page': 88, 'source': '750216638_1afcdc145e39400a9e317f4c64c179f7-100624-0453-216.pdf'}), 0.6482599377632141), (Document(page_content='Volumes which will be created as part of datastore creation will be shown in table.', metadata={'page': 88, 'source': '750216638_1afcdc145e39400a9e317f4c64c179f7-100624-0453-216.pdf'}), 0.6482599377632141)]


'Volumes which will be created as part of datastore creation will be shown in table. Volumes which will be created as part of datastore creation will be shown in table. Volumes which will be created as part of datastore creation will be shown in table. Volumes which will be created as part of datastore creation will be shown in table.'

In [ ]:
#Load the vectorstore
vectordb = Chroma(persist_directory='saved_vdb3', embedding_function=hf_embeddings)


#Get the semantic paragraph
prompt = 'traditional datastore'
search_results = vectordb.similarity_search_with_score(prompt)
print("=========vector db output =====>")
print(search_results)
context = ' '.join([result[0].page_content for result in search_results])
print("=========context is =====>")
print(context)


=========vector db output =====>
[(Document(page_content='"name": "Create traditional(NFS/VMFS) datastore: Trad_ds12_sanketh",\n      "created_time": "2024-06-13T07:28:10.256Z",\n      "modified_time": "2024-06-13T07:28:29.341Z",'), 1.2612550362673334), (Document(page_content='"name": "Create traditional(NFS/VMFS) datastore: Trad_ds12_sanketh",\n      "created_time": "2024-06-13T07:28:10.256Z",\n      "modified_time": "2024-06-13T07:28:29.341Z",'), 1.2612553834915161), (Document(page_content='"user_id": 0,\n      "description": "Create traditional(NFS/VMFS) datastore: Trad_ds12_sanketh",\n      "end_time": "2024-06-13T07:28:29.341Z",'), 1.2673057225905893), (Document(page_content='"user_id": 0,\n      "description": "Create traditional(NFS/VMFS) datastore: Trad_ds12_sanketh",\n      "end_time": "2024-06-13T07:28:29.341Z",'), 1.2673062086105347)]
=========context is =====>
"name": "Create traditional(NFS/VMFS) datastore: Trad_ds12_sanketh",
      "created_time": "2024-06-13T07:28:10.256